In [1]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Audio, display

load_dotenv()

CARTESIA_API_KEY = os.getenv("CARTESIA_API_KEY")
CARTESIA_VERSION = "2025-04-16"
CARTESIA_MODEL_ID = "sonic-3"
CARTESIA_VOICE_ID = "6ccbfb76-1fc6-48f7-b71d-91ac6298247b"
CARTESIA_LANGUAGE = "en"


## Cartesia (TTS Bytes)

Voice library: https://play.cartesia.ai/voices?tags=Emotive


In [2]:
def cartesia_list_voices(limit: int = 10):
    if not CARTESIA_API_KEY:
        raise ValueError("Missing CARTESIA_API_KEY in notebooks/.env")
    headers = {
        "Cartesia-Version": CARTESIA_VERSION,
        "Authorization": f"Bearer {CARTESIA_API_KEY}",
    }
    r = requests.get("https://api.cartesia.ai/voices", headers=headers, params={"limit": limit}, timeout=30)
    r.raise_for_status()
    data = r.json().get("data", [])
    for v in data:
        print(v.get("id"), "-", v.get("name"))
    return data


In [3]:
def cartesia_tts_bytes(text: str, output_path: str | None = "outputs/cartesia.wav") -> bytes:
    if not CARTESIA_API_KEY:
        raise ValueError("Missing CARTESIA_API_KEY in notebooks/.env")

    payload = {
        "model_id": CARTESIA_MODEL_ID,
        "transcript": text,
        "voice": {
            "mode": "id",
            "id": CARTESIA_VOICE_ID,
        },
        "output_format": {
            "container": "wav",
            "encoding": "pcm_f32le",
            "sample_rate": 24000,
        },
        "language": CARTESIA_LANGUAGE,
    }
    headers = {
        "Cartesia-Version": CARTESIA_VERSION,
        "Authorization": f"Bearer {CARTESIA_API_KEY}",
        "Content-Type": "application/json",
    }

    r = requests.post("https://api.cartesia.ai/tts/bytes", json=payload, headers=headers, timeout=60)
    r.raise_for_status()

    audio = r.content
    if output_path:
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(audio)
    return audio


In [4]:
voices = cartesia_list_voices(limit=10)
len(voices)


e07c00bc-4134-4eae-9ea4-1a55fb45746b - Brooke - Big Sister
f786b574-daa5-4673-aa0c-cbe3e8534c02 - Katie - Friendly Fixer
9626c31c-bec5-4cca-baa8-f8ba9e84c8bc - Jacqueline - Reassuring Agent
f9836c6e-a0bd-460e-9d3c-f7299fa60f94 - Caroline - Southern Guide
5ee9feff-1265-424a-9d7f-8e4d431a12c7 - Ronald - Thinker
a167e0f3-df7e-4d52-a9c3-f949145efdab - Blake - Helpful Agent
faf0731e-dfb9-4cfc-8119-259a79b27e12 - Riya - College Roommate
e8e5fffb-252c-436d-b842-8879b84445b6 - Cathy - Coworker
95d51f79-c397-46f9-b49a-23763d3eaa2d - Arushi - Hinglish Speaker
79f8b5fb-2cc8-479a-80df-29f7a7cf1a3e - Theo - Modern Narrator


10

In [5]:
audio = cartesia_tts_bytes("Hello from Cartesia. This is a quick TTS smoke test.")
len(audio)


428792

In [6]:
display(Audio(audio))
